# Tornado — بناء قاعدة المعرفة المعجمية

شغّل الخلايا بالترتيب. المُخرَج ملفٌ واحد: **`tornado-kb.sqlite`**.

| المرحلة | الزمن التقريبي |
|---|---|
| تثبيت المكتبات | ~٢ دقيقة |
| ويكاموس + WordNet + Tatoeba | ~٤٠ دقيقة |
| المتلازمات (`dependency`) | ~٣٠ دقيقة |

**لا يُطلب منك مفتاح ولا حساب ولا رفع ملفات.** القوائم تُقرأ من مستودعك العام مباشرة.

> إن انقطعت الجلسة: أعد التشغيل من الخلية ٣ — التنزيلات محفوظة ولا تُعاد.

### ١ — المكتبات

In [ ]:
!pip -q install datasets spacy nltk
!python -m spacy download en_core_web_sm -q
print('تمّ')

### ٢ — حفظ العمل في Drive (اختياري لكن مُستحسَن)

جلسة Colab تُمسح عند الانقطاع. الربط بـ Drive يجعل التنزيلات الكبيرة تنجو،
فلا تعيد تنزيل غيغابايتات لو انقطعت الجلسة في منتصف الطريق.

In [ ]:
import os
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['TORNADO_WORK'] = '/content/drive/MyDrive/tornado-kb'
else:
    os.environ['TORNADO_WORK'] = '/content/kb'

os.makedirs(os.environ['TORNADO_WORK'], exist_ok=True)
print('مجلد العمل:', os.environ['TORNADO_WORK'])

### ٣ — البناء

المنطق كله في `build_kb.py` داخل المستودع — مقروء ومُراجَع هناك، لا مدفون في خلايا.

لإعادة تشغيل مرحلة واحدة فقط، اضبط قبل التشغيل مثلاً:
`os.environ['TORNADO_STAGES'] = 'collocations'`

In [ ]:
import os, urllib.request

SRC = 'https://raw.githubusercontent.com/drtornado/tornado/main/tools/enrich/build_kb.py'
FORCE_REFRESH = False   # True = تجاهل النسخة المحلية واجلب أحدث نسخة

# المراحل: all | vocab | wiktionary | wordnet | tatoeba | collocations
# بعد فشلٍ في منتصف الطريق، اذكر المتبقّي وحده — المنجَز لا يُعاد.
STAGES = 'all'

# ── المتلازمات ──────────────────────────────────────────────────────
# 'window'     : بلا تحليل نحوي — دقائق. يكفي تماماً للحكم على الجودة أوّلاً.
# 'dependency' : فعل + مفعوله فعلاً — أدقّ، لكنه بطيء على معالجَي Colab.
COLLOC = 'window'
COLLOC_TOKENS = '5000000'
COLLOC_MAX_MIN = '25'      # سقفٌ صارم: عنده يُحسب ما جُمع بدل الانتظار المفتوح
COLLOC_PROCS = '1'         # 2 يخنق العملية الأمّ على Colab المجاني

if FORCE_REFRESH and os.path.exists('build_kb.py'):
    os.remove('build_kb.py')

# لو لم يُرفع tools/enrich/ إلى المستودع بعد، ارفع build_kb.py يدوياً من أيقونة
# المجلد على يسار Colab — عندئذٍ تُستعمل النسخة المحلية بدل أن تفشل الخلية بـ 404.
if os.path.exists('build_kb.py'):
    print('أستعمل النسخة المحلية')
else:
    try:
        urllib.request.urlretrieve(SRC, 'build_kb.py')
        print('جُلب من المستودع')
    except Exception as e:
        raise SystemExit(
            f'تعذّر الجلب ({e}).\n'
            'إمّا أن ترفع tools/enrich/ إلى المستودع، أو ترفع build_kb.py\n'
            'يدوياً من أيقونة المجلد على يسار Colab ثم تعيد تشغيل هذه الخلية.')

# تأكيدٌ صريح على النسخة — بدونه قد تُعاد تشغيل نسخة قديمة ويُظنّ الإصلاح فاشلاً
for ln in open('build_kb.py', encoding='utf-8'):
    if ln.startswith('VERSION'):
        print('النسخة:', ln.split('=', 1)[1].strip().strip('"'))
        break

# إسنادٌ صريح لا setdefault: القيمة تبقى من التشغيلة السابقة في نفس الجلسة،
# فيُعاد بناء ويكاموس ساعةً كاملة بلا داعٍ.
os.environ['TORNADO_STAGES'] = STAGES
os.environ['TORNADO_COLLOC'] = COLLOC
os.environ['TORNADO_COLLOC_TOKENS'] = COLLOC_TOKENS
os.environ['TORNADO_COLLOC_MAX_MIN'] = COLLOC_MAX_MIN
os.environ['TORNADO_COLLOC_PROCS'] = COLLOC_PROCS
print(f'المراحل: {STAGES} · متلازمات: {COLLOC} × {int(COLLOC_TOKENS):,}')

# التنزيلات الموجودة لا تُعاد — إعادة التشغيل بعد خطأ تكلّف المعالجة فقط
!python build_kb.py

### ٣٫٥ — تدقيق ثم إصلاح (ثوانٍ، بلا إعادة بناء)

**أوّلاً إثبات، ثم إصلاح.** `audit()` يقارن `COUNT(*)` بعدد الصفوف الفريدة لكل جدول:

- نسبة **١٫٠٠×** ⟵ لا تكرار في القاعدة، والعيب في العرض وحده
- نسبة **٢٫٠٠×** ⟵ المرحلة جرت مرّتين، والتكرار داخل القاعدة

ثم `repair()` يحذفه بـ SQL في ثوانٍ — **لا حاجة لإعادة تشغيل أي مرحلة.**

In [ ]:
# ═══ تحقّق من الملف قبل أي إصلاح ═══
# لا تثق بـ import: يقرأ من القرص، وقد يكون على القرص ملفٌّ قديم.
import os, sys, time, hashlib

EXPECT = {'version': '7', 'bytes': 70389, 'md5': '693ccbbbe7f52683',
          'funcs': ['audit', 'repair', 'full_report', 'sample_cards',
                    'inspect_wiktextract', 'build_card', 'ar_pron']}

print('المجلد الحالي:', os.getcwd())
print('\nنسخ build_kb.py على القرص:')
found = []
for root in ('.', '/content', '/content/drive/MyDrive/tornado-kb',
             os.environ.get('TORNADO_WORK', '/content/kb')):
    p = os.path.abspath(os.path.join(root, 'build_kb.py'))
    if os.path.exists(p) and p not in found:
        found.append(p)
        st = os.stat(p)
        print(f'  {p}\n     {st.st_size:,} بايت · '
              f'{time.strftime("%H:%M:%S", time.localtime(st.st_mtime))}')
if not found:
    raise SystemExit('لا يوجد build_kb.py إطلاقاً — ارفعه أوّلاً')

# ── الملف الذي سيُستورد فعلاً ──
raw = open('build_kb.py', 'rb').read()
src = raw.decode('utf-8')
print(f'\nالملف المستورَد: build_kb.py · {len(raw):,} بايت'
      f' · md5 {hashlib.md5(raw).hexdigest()[:16]}')
print(f'  متوقّع: {EXPECT["bytes"]:,} بايت · md5 {EXPECT["md5"]}')
if len(raw) != EXPECT['bytes']:
    print('  ⚠ الحجم مختلف — قد يكون النقل بدّل نهايات الأسطر، '
          'والحكم الفاصل هو VERSION والدوال أدناه')

ver = next((l for l in src.splitlines() if l.startswith('VERSION')), None)
print(f'\nVERSION في الملف: {ver or "← غير موجود إطلاقاً (نسخة قديمة جداً)"}')

print('\nالدوال داخل نصّ الملف:')
missing = [f for f in EXPECT['funcs'] if f'\ndef {f}(' not in src]
for fn in EXPECT['funcs']:
    print(f'  def {fn:<22} {"موجود ✅" if fn not in missing else "مفقود ❌"}')

# ── استيرادٌ نظيف: reload لا يكفي إن كانت الوحدة محمّلة من مسار آخر ──
sys.modules.pop('build_kb', None)
import build_kb

print(f'\nمسار الوحدة المستوردة: {build_kb.__file__}')
print(f'النسخة من الوحدة     : {getattr(build_kb, "VERSION", "← لا يوجد")}')
have = [f for f in EXPECT['funcs'] if hasattr(build_kb, f)]
print(f'الدوال المتاحة       : {len(have)}/{len(EXPECT["funcs"])}')

if missing or not hasattr(build_kb, 'audit'):
    raise SystemExit(
        '\n❌ الملف قديم. الحلّ:\n'
        '   ارفع build_kb.py الجديد (أيقونة المجلد يمين ← Upload ← Replace)\n'
        '   وتأكّد أنه في /content لا في مجلد آخر، ثم أعد تشغيل هذه الخلية.\n'
        '   أو ادفعه إلى المستودع واضبط FORCE_REFRESH = True في خلية البناء.')
print('\n✅ النسخة صحيحة — امضِ إلى التدقيق')

In [ ]:
import importlib, build_kb
importlib.reload(build_kb)

before = build_kb.audit()        # الإثبات
build_kb.repair(apply=False)     # ماذا سيُحذف — بلا كتابة

# اقرأ ما سبق، ثم شغّل الخليتين التاليتين للتنفيذ والتحقّق

In [ ]:
build_kb.repair(apply=True)      # التنفيذ
after = build_kb.audit()         # التحقّق: كل النسب يجب أن تصير 1.00×

### ٤ — التقرير الفعلي

أرقام **مقيسة من القاعدة**، لا تقديرات. الحقول المعلَّمة `▸` هي السبعة المطلوبة صراحةً.

In [ ]:
import importlib, build_kb
importlib.reload(build_kb)          # يلتقط أي تعديل على الملف بلا إعادة تشغيل

report = build_kb.full_report()

# لماذا Usage Notes صفر؟ نقرأ أسماء الحقول من الملف بدل تخمينها.
# يمرّ على ٣ ملايين سطر — دقيقة أو نحوها.
fields = build_kb.inspect_wiktextract()

### ٥ — عشر بطاقات حقيقية

مولَّدة من القاعدة لكلمات **من مكتبتك أنت**، لا أمثلة مختارة.

العيّنة موزّعة على مدى الشيوع عمداً: كلمة شائعة وأخرى نادرة تكشفان التفاوت —
وهو ما يُخفيه انتقاء الأسهل.

> يُنتَج أيضاً `cards.html` — احكم عليه بالعين، لا على JSON.

In [ ]:
cards = build_kb.sample_cards(n=10)

# لفحص كلمات بعينها بدلاً من العيّنة الموزّعة:
# cards = build_kb.sample_cards(words=['issue', 'assume', 'abide'])

from IPython.display import HTML, display
display(HTML(open('/content/cards.html', encoding='utf-8').read()))

### ٦ — التنزيل

نزّل القاعدة **وملف البطاقات** معاً. البطاقات هي ما تحكم عليه قبل بناء Card Builder.

**لن تعود إلى هذا الدفتر مرّة أخرى** — التغطية أوسع من أي كلمة ستضيفها لاحقاً.

In [ ]:
import os, shutil, json
from google.colab import files

WORK = os.environ['TORNADO_WORK']
DB = os.path.join(WORK, 'tornado-kb.sqlite')

# تقرير نصّي يبقى معك للمقارنة بعد أي إعادة بناء
with open(os.path.join(WORK, 'report.json'), 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=1)

shutil.copy('/content/cards.html', WORK)
print(f'القاعدة: {os.path.getsize(DB)/1e6:.0f} م.ب')

# الضغط يقلّص الحجم كثيراً — قواعد SQLite نصّية في جوهرها
shutil.make_archive('/content/tornado-kb', 'zip', WORK)
print(f'المجموع مضغوطاً: {os.path.getsize("/content/tornado-kb.zip")/1e6:.0f} م.ب')

files.download('/content/cards.html')     # البطاقات أوّلاً — هي محلّ الحكم
files.download('/content/tornado-kb.zip')